In [1]:
%%capture
!pip3 install vosk

In [2]:
from vosk import Model, KaldiRecognizer
import wave
import json

In [13]:
import IPython
IPython.display.Audio("/tmp/tmp.wav")

In [15]:
model = Model(model_path="/Users/joregan/vosk/vosk-model-en-us-0.42-gigaspeech/")

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=8
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /Users/joregan/vosk/vosk-model-en-us-0.42-gigaspeech//ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from /Users/joregan/vosk/vosk-model-en-us-0.42-gigaspeech//graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:294) Loading words from /Users/joregan/vosk/vosk-model-en-us-0.42-gigaspeech//graph/words.txt
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo /Users/joregan/vosk/v

In [16]:
def format_ctm(data, filename, offset):
    for res in data["result"]:
        start = res["start"] + offset
        end = res["end"] + offset
        word = res["word"]
        conf = res["conf"]
        print(f"{filename} 1 {start:.02f} {end:.02f} {conf} {word} 1.0 {word} cor")

In [11]:
INPUT = """
hsi_5_0718_209_001_main 1 32.47 31.68 ah 1.0 Ah, sub
hsi_5_0718_209_001_main 1 32.46 32.58 you 1.0 you cor
hsi_5_0718_209_001_main 1 32.58 32.76 see 1.0 see cor
hsi_5_0718_209_001_main 1 32.76 33.0 this 1.0 this cor
hsi_5_0718_209_001_main 1 33.0 33.42 couch 1.0 couch cor
hsi_5_0718_209_001_main 1 33.48 33.611924 and 0.582101 and cor
hsi_5_0718_209_001_main 1 33.611924 33.81 and 1.0 and cor
hsi_5_0718_209_001_main 1 33.81 34.23 the_what 1.0 the_what sub
hsi_5_0718_209_001_main 1 34.92 35.37 do_you_call 1.0 do_you_call sub
hsi_5_0718_209_001_main 1 35.37 35.58 this 1.0 this cor
hsi_5_0718_209_001_main 1 35.58 35.76 one 1.0 one cor
hsi_5_0718_209_001_main 1 35.76 35.85 in 1.0 in cor
hsi_5_0718_209_001_main 1 35.85 36.27 swedish 1.0 Swedish cor
hsi_5_0718_209_001_main 1 36.45 36.6 or 1.0 or cor
hsi_5_0718_209_001_main 1 36.6 36.780527 in 1.0 in cor
hsi_5_0718_209_001_main 1 36.780527 37.23 english 0.975392 English sub
hsi_5_0718_209_001_main 1 37.23 37.8 fotolia 1.0 fåtölj sub
hsi_5_0718_209_001_main 1 38.94 39.33 chair 1.0 chair cor
hsi_5_0718_209_001_main 1 39.33 39.72 yes 1.0 yes cor
hsi_5_0718_209_001_main 1 39.72 39.96 this 1.0 this- ins
hsi_5_0718_209_001_main 1 39.99 40.56 armchair 0.419268 armchair, sub
hsi_5_0718_209_001_main 1 40.56 40.92 yes 1.0 yes, cor
hsi_5_0718_209_001_main 1 40.95 41.16 this 1.0 this cor
hsi_5_0718_209_001_main 1 41.16 41.73 armchair 1.0 armchair cor
hsi_5_0718_209_001_main 1 42.48 43.05 ah 1.0 ah sub
hsi_5_0718_209_001_main 1 43.32 43.83 is 1.0 is cor
hsi_5_0718_209_001_main 1 43.89 44.49 really 1.0 really cor
hsi_5_0718_209_001_main 1 44.49 45.18 really 1.0 really cor
hsi_5_0718_209_001_main 1 45.24 45.84 amazing 1.0 amazing cor
hsi_5_0718_209_001_main 1 45.84 46.2 because 1.0 because cor
hsi_5_0718_209_001_main 1 46.2 46.35 it's 1.0 it's cor
hsi_5_0718_209_001_main 1 46.35 46.41 a 1.0 a cor
hsi_5_0718_209_001_main 1 46.41 46.83 danish 1.0 Danish cor
hsi_5_0718_209_001_main 1 46.83 47.49 designer 1.0 designer cor
hsi_5_0718_209_001_main 1 48.0 48.81 who's 0.919197 who's cor
hsi_5_0718_209_001_main 1 49.14 49.44 really 0.997848 really cor
hsi_5_0718_209_001_main 1 49.44 49.98 famous 1.0 famous. cor
"""

In [4]:
def get_ffmpeg_cmd(ctmeditlines):
    startline = ctmeditlines[0]
    endline = ctmeditlines[-1]
    start = startline.split()[2]
    end = endline.split()[3]
    filename = startline.split()[0]
    return f"ffmpeg -i {filename}.wav -ss {start} -t {float(end) - float(start)} -acodec pcm_s16le -ac 1 -ar 16000 /tmp/tmp.wav"

In [8]:
ctmlines = [x for x in "hsi_5_0718_209_001_inter 1 60.147906 61.768000 Which vases do you mean?".split("\n") if x != ""]

In [9]:
get_ffmpeg_cmd(ctmlines)

'ffmpeg -i hsi_5_0718_209_001_inter.wav -ss 60.147906 -t 1.6200940000000017 -acodec pcm_s16le -ac 1 -ar 16000 /tmp/tmp.wav'

In [15]:
text = "'[\"" + " ".join([x.split()[4] for x in ctmlines]).replace("_", " ").replace("'", "\\'") + "\"]'"

In [17]:
wf = wave.open('/tmp/tmp.wav', "rb")
rec = KaldiRecognizer(model, wf.getframerate(), '["which vases do you mean"]')
rec.SetWords(True)

while True:
    data = wf.readframes(4000)
    if len(data) == 0:
        break
    if rec.AcceptWaveform(data):
        print(rec.Result())
    else:
        jres = json.loads(rec.PartialResult())
        print(jres)

final = rec.FinalResult()

WARNING (VoskAPI:Recognizer():recognizer.cc:97) Runtime graphs are not supported by this model


{'partial': ''}
{'partial': ''}
{'partial': ''}
{'partial': ''}
{'partial': ''}
{'partial': 'which'}


In [25]:
format_ctm(json.loads(final), "hsi_5_0718_209_001_inter", 60.4)

hsi_5_0718_209_001_inter 1 60.82 61.06 1.0 which 1.0 which cor
hsi_5_0718_209_001_inter 1 61.06 61.39 1.0 was 1.0 was cor
hsi_5_0718_209_001_inter 1 61.39 61.48 0.845489 do 1.0 do cor
hsi_5_0718_209_001_inter 1 61.48 61.57 0.845489 you 1.0 you cor
hsi_5_0718_209_001_inter 1 61.57 61.78 0.845489 mean 1.0 mean cor
